In [1]:
# %%
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.metrics import mean_squared_error, mean_absolute_error
from statsmodels.tsa.statespace.sarimax import SARIMAX

# ---- Make repo importable (so `configs...` works) ----
repo_root = Path.cwd().resolve()
while repo_root.name != "graphCeylon-API" and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from configs.recommender_service.config import WEATHER_RAW_DIR, DATA_CLEAN_DIR

HISTORY_FILE = WEATHER_RAW_DIR / "cleaned_history.csv"
OUT_FORECAST = DATA_CLEAN_DIR / "sarimax_rainfall_forecast.csv"
OUT_SCORES   = DATA_CLEAN_DIR / "sarimax_rainfall_model_scores.csv"

print("History:", HISTORY_FILE)
print("Exists? ", HISTORY_FILE.exists())


History: C:\Users\HP\OneDrive\Documents\graphCeylon-API\data\recommender_service\raw\data_raw\weather\cleaned_history.csv
Exists?  True


In [2]:
# %%
if not HISTORY_FILE.exists():
    raise FileNotFoundError(f"History file not found: {HISTORY_FILE}")

df = pd.read_csv(HISTORY_FILE)

# location_id normalize
df["location_id"] = (
    df["location_id"].astype(str)
      .str.lower()
      .str.strip()
      .str.replace("’", "'", regex=False)
      .str.replace("'", "", regex=False)
      .str.replace("-", "_", regex=False)
      .str.replace(" ", "_", regex=False)
)

# Parse date (do NOT parse twice later)
df["date"] = pd.to_datetime(df["date"], errors="coerce")

df = df.dropna(subset=["date", "location_id", "rainfall"]).copy()
df["rainfall"] = pd.to_numeric(df["rainfall"], errors="coerce").fillna(0.0).clip(lower=0)

if "avg_temp" in df.columns:
    df["avg_temp"] = pd.to_numeric(df["avg_temp"], errors="coerce")

print(df.head())
print("Locations:", df["location_id"].nunique())


        date location_id  avg_temp  rainfall
0 2021-01-01  adams_peak     15.55      91.4
1 2021-02-01  adams_peak     15.40      59.2
2 2021-03-01  adams_peak     16.15     108.7
3 2021-04-01  adams_peak     16.60     107.2
4 2021-05-01  adams_peak     17.20     273.2
Locations: 38


In [3]:
# %%
d0 = pd.to_datetime(df["date"], errors="coerce", dayfirst=False)
d1 = pd.to_datetime(df["date"], errors="coerce", dayfirst=True)

score0 = (d0.dt.day == 1).mean()
score1 = (d1.dt.day == 1).mean()

df["date"] = d0 if score0 >= score1 else d1
print("Picked dayfirst =", (score1 > score0), " | day==1 ratio:", max(score0, score1))

df = df.dropna(subset=["date"]).copy()


Picked dayfirst = False  | day==1 ratio: 1.0


In [10]:
# %%
def ensure_monthly(loc_df: pd.DataFrame) -> pd.DataFrame:
    loc_df = loc_df.sort_values("date").copy()

    # Convert any date to month-start timestamp safely
    loc_df["date"] = loc_df["date"].dt.to_period("M").dt.start_time

    agg = {"rainfall": "sum"}  # monthly total
    if "avg_temp" in loc_df.columns:
        agg["avg_temp"] = "mean"

    loc_df = (
        loc_df.groupby(["location_id", "date"], as_index=False)
              .agg(agg)
              .sort_values("date")
    )

    start = loc_df["date"].min()
    end   = loc_df["date"].max()
    idx = pd.date_range(start, end, freq="MS")

    loc_df = loc_df.set_index("date").reindex(idx)
    loc_df.index.name = "date"

    loc_df["location_id"] = loc_df["location_id"].ffill().bfill()
    loc_df["rainfall"] = loc_df["rainfall"].fillna(0.0).clip(lower=0)

    if "avg_temp" in loc_df.columns:
        loc_df["avg_temp"] = (
            loc_df["avg_temp"]
            .interpolate(limit_direction="both")
            .bfill()
            .ffill()
        )

    return loc_df.reset_index()   # ✅ MUST be here
_tmp = df[df["location_id"] == sorted(df["location_id"].unique())[0]].copy()
_tmp = ensure_monthly(_tmp)

print(type(_tmp))
print(_tmp.shape)
print(_tmp.head())

def make_exog(dates: pd.Series, avg_temp: pd.Series | None = None) -> pd.DataFrame:
    # IMPORTANT: reset index so SARIMAX aligns exog correctly
    dates = pd.to_datetime(dates).reset_index(drop=True)

    m = dates.dt.month
    exog = pd.DataFrame({
        "month_sin": np.sin(2*np.pi*m/12),
        "month_cos": np.cos(2*np.pi*m/12),
    })

    if avg_temp is not None:
        exog["avg_temp"] = pd.Series(avg_temp).reset_index(drop=True).values

    return exog


def transform_rain(y_mm, cap_q=0.98):
    y = pd.Series(y_mm).astype(float).clip(lower=0)
    cap = y.quantile(cap_q)
    y_cap = y.clip(upper=cap)
    return np.log1p(y_cap).values, float(cap)


def inv_transform_rain(y_log, cap):
    y_mm = np.expm1(np.asarray(y_log))
    return np.clip(y_mm, 0, cap)


def rmse_mm(y_true_mm, y_pred_mm):
    return float(np.sqrt(mean_squared_error(y_true_mm, y_pred_mm)))


def smape(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred) + eps)
    return float(100.0 * np.mean(2.0 * np.abs(y_pred - y_true) / denom))


def peak_rmse(y_true, y_pred, k=3):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    k = min(k, len(y_true))
    idx = np.argsort(y_true)[-k:]
    return float(np.sqrt(mean_squared_error(y_true[idx], y_pred[idx])))


<class 'pandas.core.frame.DataFrame'>
(58, 4)
        date location_id  rainfall  avg_temp
0 2021-01-01  adams_peak     182.8     15.55
1 2021-02-01  adams_peak     118.4     15.40
2 2021-03-01  adams_peak     217.4     16.15
3 2021-04-01  adams_peak     214.4     16.60
4 2021-05-01  adams_peak     546.4     17.20


In [11]:
# %%
P = [0, 1, 2]
D = [0, 1]
Q = [0, 1, 2]

SP = [0, 1]
SD = [0, 1]
SQ = [0, 1]
S  = 12  # monthly

def fit_eval_sarimax(loc_df: pd.DataFrame, test_horizon=12, cap_q=0.98):
    loc_df = loc_df.sort_values("date").copy()

    if len(loc_df) < (test_horizon + 18):
        test_horizon = max(6, len(loc_df) // 4)

    cut = len(loc_df) - test_horizon
    train_df = loc_df.iloc[:cut].copy()
    test_df  = loc_df.iloc[cut:].copy()

    train_y_log, cap = transform_rain(train_df["rainfall"].values, cap_q=cap_q)
    test_y_mm = test_df["rainfall"].values

    use_temp = "avg_temp" in loc_df.columns and loc_df["avg_temp"].notna().any()

    train_exog = make_exog(train_df["date"], train_df["avg_temp"] if use_temp else None)
    test_exog  = make_exog(test_df["date"],  test_df["avg_temp"] if use_temp else None)

    best_rmse = float("inf")
    best_res = None
    best_cfg = None
    best_metrics = None

    for p in P:
        for d in D:
            for q in Q:
                for sp in SP:
                    for sd in SD:
                        for sq in SQ:
                            order = (p, d, q)
                            seasonal_order = (sp, sd, sq, S)

                            try:
                                model = SARIMAX(
                                    train_y_log,
                                    exog=train_exog,
                                    order=order,
                                    seasonal_order=seasonal_order,
                                    enforce_stationarity=False,
                                    enforce_invertibility=False,
                                )
                                res = model.fit(disp=False, maxiter=300)

                                pred_log = res.get_forecast(
                                    steps=len(test_df),
                                    exog=test_exog
                                ).predicted_mean

                                pred_mm = inv_transform_rain(pred_log, cap)

                                rmse = rmse_mm(test_y_mm, pred_mm)
                                if not np.isfinite(rmse):
                                    continue

                                metrics = {
                                    "rmse": rmse,
                                    "mae": float(mean_absolute_error(test_y_mm, pred_mm)),
                                    "smape": smape(test_y_mm, pred_mm),
                                    "peak_rmse": peak_rmse(test_y_mm, pred_mm),
                                    "aic": float(res.aic),
                                    "bic": float(res.bic),
                                    "converged": bool(res.mle_retvals.get("converged", True)),
                                }

                                if rmse < best_rmse:
                                    best_rmse = rmse
                                    best_res = res
                                    best_cfg = {
                                        "order": order,
                                        "seasonal_order": seasonal_order,
                                        "use_temp": use_temp,
                                        "cap": cap,
                                        "cap_q": cap_q,
                                    }
                                    best_metrics = metrics

                            except Exception:
                                continue

    return best_res, best_cfg, best_metrics, train_df, test_df


In [12]:
# %%
forecast_horizon = 24
test_horizon = 12

all_fc = []
scores = []

locations = sorted(df["location_id"].unique())

for loc in locations:
    loc_df = df[df["location_id"] == loc].copy()
    loc_df = ensure_monthly(loc_df)

    if len(loc_df) < 18:
        print(f"⚠️ Skipping {loc} (too little data: {len(loc_df)})")
        continue

    print(f"📍 SARIMAX: {loc} ({len(loc_df)} months)")

    best_res, best_cfg, metrics, train_df, test_df = fit_eval_sarimax(loc_df, test_horizon=test_horizon)

    if best_res is None or best_cfg is None or metrics is None:
        print(f"   ❌ No model converged for {loc}")
        continue

    # Refit on full data using SAME transform (cap_q)
    full_y_log, cap = transform_rain(loc_df["rainfall"].values, cap_q=best_cfg["cap_q"])

    use_temp = best_cfg["use_temp"]
    full_exog = make_exog(loc_df["date"], loc_df["avg_temp"] if (use_temp and "avg_temp" in loc_df.columns) else None)

    refit = SARIMAX(
        full_y_log,
        exog=full_exog,
        order=best_cfg["order"],
        seasonal_order=best_cfg["seasonal_order"],
        enforce_stationarity=False,
        enforce_invertibility=False,
    ).fit(disp=False, maxiter=300)

    last_date = loc_df["date"].max()
    future_dates = pd.date_range(last_date + pd.offsets.MonthBegin(1), periods=forecast_horizon, freq="MS")

    if use_temp:
        temp_monthly_mean = loc_df.groupby(loc_df["date"].dt.month)["avg_temp"].mean()
        future_temp = pd.Series([temp_monthly_mean[d.month] for d in future_dates])
        future_exog = make_exog(pd.Series(future_dates), future_temp)
    else:
        future_exog = make_exog(pd.Series(future_dates), None)

    pred_log = refit.get_forecast(steps=forecast_horizon, exog=future_exog).predicted_mean
    pred_mm = inv_transform_rain(pred_log, cap)

    fc = pd.DataFrame({
        "ds": future_dates,
        "location_id": loc,
        "pred_rainfall": np.round(pred_mm, 1)
    })
    all_fc.append(fc)

    scores.append({
        "location_id": loc,
        "rmse_mm_last12": round(metrics["rmse"], 2),
        "mae_mm": round(metrics["mae"], 2),
        "smape_pct": round(metrics["smape"], 2),
        "peak_rmse_mm": round(metrics["peak_rmse"], 2),
        "aic": round(metrics["aic"], 2),
        "bic": round(metrics["bic"], 2),
        "order": str(best_cfg["order"]),
        "seasonal_order": str(best_cfg["seasonal_order"]),
        "used_temp_exog": bool(use_temp),
        "converged": bool(metrics["converged"]),
        "n_months": len(loc_df)
    })

# Save
DATA_CLEAN_DIR.mkdir(parents=True, exist_ok=True)

if all_fc:
    final_fc = pd.concat(all_fc, ignore_index=True)
    final_fc.to_csv(OUT_FORECAST, index=False)
    print(f"\n✅ Saved forecast -> {OUT_FORECAST}")

scores_df = pd.DataFrame(scores).sort_values("rmse_mm_last12", ascending=True)
scores_df.to_csv(OUT_SCORES, index=False)
print(f"✅ Saved scores   -> {OUT_SCORES}")

scores_df.head(15)


📍 SARIMAX: adams_peak (58 months)


c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607:

📍 SARIMAX: anuradhapura (58 months)


c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607:

📍 SARIMAX: arugambay (53 months)


...

c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607:

📍 SARIMAX: yala_national_park (33 months)


c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
c:\Users\HP\OneDrive\Documents\Proactive-Tourism-Recommendation\component3_service\venv\Lib\site-packages\statsmodels\base\model.py:607:


✅ Saved forecast -> C:\Users\HP\OneDrive\Documents\graphCeylon-API\data\recommender_service\clean\sarimax_rainfall_forecast.csv
✅ Saved scores   -> C:\Users\HP\OneDrive\Documents\graphCeylon-API\data\recommender_service\clean\sarimax_rainfall_model_scores.csv


,location_id,rmse_mm_last12,mae_mm,smape_pct,peak_rmse_mm,aic,bic,order,seasonal_order,used_temp_exog,converged,n_months
6,bundala_national_park,37.91,31.07,44.96,38.57,65.91,73.46,"(2, 1, 1)","(0, 1, 1, 12)",True,True,58
11,hambantota,37.91,31.07,44.96,38.57,65.91,73.46,"(2, 1, 1)","(0, 1, 1, 12)",True,True,58
1,anuradhapura,53.78,37.93,55.78,29.22,1548.61,1557.41,"(0, 1, 0)","(1, 0, 1, 12)",True,False,58
10,galle,58.45,48.42,29.89,36.95,60.45,65.68,"(0, 0, 0)","(0, 1, 1, 12)",True,True,58
12,hikkaduwa,58.45,48.42,29.89,36.95,60.45,65.68,"(0, 0, 0)","(0, 1, 1, 12)",True,True,58
21,matara,58.45,48.42,29.89,36.95,60.45,65.68,"(0, 0, 0)","(0, 1, 1, 12)",True,True,58
34,weligama,58.45,48.42,29.89,36.95,60.45,65.68,"(0, 0, 0)","(0, 1, 1, 12)",True,True,58
33,unawatuna,58.45,48.42,29.89,36.95,60.45,65.68,"(0, 0, 0)","(0, 1, 1, 12)",True,True,58
16,kalpitiya,63.89,36.96,95.12,111.36,83.44,90.94,"(1, 1, 2)","(1, 1, 1, 12)",True,True,57
18,katunayake,64.01,51.60,42.60,82.89,92.99,99.53,"(0, 0, 1)","(1, 1, 0, 12)",True,True,58
